# Coordinate Predictor

## Imports

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
from PIL import Image
import argparse
from tqdm.auto import tqdm
import random
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF
import logging
from typing import Callable, Optional
import wandb

In [36]:
def set_seed(seed: int = 32) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

set_seed()

## Dataset Creation

In [19]:
class CreateDataset:
    """
    this class generates all possible 50x50 grayscale images where only one pixel has 255 value & rest all have 0.
    at the end we will have 2500 images with each image having unique id & a json file which maps id to coordinates.
    """
    def __init__(
            self,
            image_size: int,
            dataset_dir : str,
            train_split: float,
            random_seed: int = 42
    ) -> None:

        self.image_size = image_size

        self.dataset_dir = dataset_dir
        os.makedirs(self.dataset_dir, exist_ok=True)

        self.image_dir = os.path.join(self.dataset_dir, "images")
        os.makedirs(self.image_dir, exist_ok=True)

        self.train_len = int(train_split * len(self))
        val_test_split = round((1 - train_split) / 2, 1)
        self.val_len = self.train_len + int(val_test_split * len(self))

        self.train_dir = os.path.join(self.image_dir, "train")
        self.val_dir = os.path.join(self.image_dir, "val")
        self.test_dir = os.path.join(self.image_dir, "test")

        os.makedirs(self.train_dir, exist_ok=True)
        os.makedirs(self.val_dir, exist_ok=True)
        os.makedirs(self.test_dir, exist_ok=True)

        np.random.seed(random_seed)

    def __len__(self) -> int:
        return self.image_size ** 2

    def create_samples(self) -> None:

        # creating a list of possible coordinate values &
        # randomly shuffling it so that we have randomly shuffled values of coordinates.
        coordinates_list = []
        for i in range(self.image_size):
            for j in range(self.image_size):
                coordinates_list.append([i, j])

        random.shuffle(coordinates_list)

        target_coordinate_dict = {}

        # Initializing image only once & just turning pixels on & off for every sample.
        image = np.zeros((self.image_size, self.image_size), dtype=np.uint8)
        for i in tqdm(range(len(coordinates_list))):
            x, y = coordinates_list[i]
            sample_name = f"{i:06}"

            image[y, x] = 255
            pil_image = Image.fromarray(image)
            if i < self.train_len:
                pil_image.save(os.path.join(self.train_dir, f"{sample_name}.png"))
            elif i < self.val_len:
                pil_image.save(os.path.join(self.val_dir, f"{sample_name}.png"))
            else:
                pil_image.save(os.path.join(self.test_dir, f"{sample_name}.png"))
            image[y, x] = 0

            target_coordinate_dict[sample_name] = [x, y]

        json_file_path = os.path.join(self.dataset_dir, "target_coordinates.json")
        with open(json_file_path, "w") as f:
            json.dump(target_coordinate_dict, f, indent=4)

In [20]:
create_dataset = CreateDataset(
        image_size = 50,
        dataset_dir = "Coordinate_Dataset",
        train_split = 0.8,
        random_seed = 42
    )
create_dataset.create_samples()

  0%|          | 0/2500 [00:00<?, ?it/s]

In [21]:
class CoordinateDataset(torch.utils.data.Dataset):
    def __init__(
            self,
            image_dir : str,
            coordinates_json_path: str,
            device: str,
            transforms: Optional[Callable] = None
    ) -> None:
        assert os.path.isdir(image_dir), f"Image directory {image_dir} nor found"
        assert os.path.isfile(coordinates_json_path), f"json file {coordinates_json_path} not found"
        self.image_dir = image_dir
        self.transforms = transforms

        image_list = os.listdir(image_dir)
        self.image_list = [image[:-4] for image in image_list]  # gets list of image ids

        with open(coordinates_json_path, "r") as f:
            self.coordinates_dict = json.load(f)

    def __len__(self) -> None:
        return len(self.image_list)

    def __getitem__(self, idx) -> dict:
        sample_name = self.image_list[idx]
        coordinates = np.array(self.coordinates_dict[sample_name])
        coordinates = coordinates / 50
        try:
            image = Image.open(os.path.join(self.image_dir, f"{sample_name}.png"))
            if self.transforms:
                image = self.transforms(image)
                # image /= 255.0
            return {
                'image' : image,
                'coordinates' : coordinates
            }
        except (FileNotFoundError, IOError) as e:
            logging.error(f"Could not read files for index {idx}. File: {e.filename}. Skipping.")
            return self.__getitem__((idx + 1) % self.__len__())   # Increments index and tries again with the next valid pair


def get_dataloaders(
        dataset_dir: str,
        batch_size: int,
        shuffle: bool,
        device: str
) -> dict:
    """
    gives a dictionary of dataloaders for train, val & test

    Args:
        dataset_dir (str): directory path for the dataset
        batch_size (int): batch size of the dataloaders
        shuffle (bool): to shuffle or not
        device (str): device

    Returns:
        dataloaders (dict): dictionary of dataloaders for train, val & test
    """
    coordinates_json_path = os.path.join(dataset_dir, "target_coordinates.json")
    datasets = {x : CoordinateDataset(os.path.join(dataset_dir, "images", x), coordinates_json_path, device, transforms.ToTensor())
                for x in ['train', 'val', 'test']}

    dataloaders = {x : torch.utils.data.DataLoader(datasets[x], batch_size, shuffle)
                   for x in ['train', 'val', 'test']}

    return dataloaders

## Models

In [22]:
class CoordinateRegressor(nn.Module):
    """CNN architecture for predicting (x, y) coordinates of a pixel."""

    def __init__(self, input_shape:int, mlp_units:int, output_shape:int, image_size: int = 50, dropout: float = 0.3):
        super(CoordinateRegressor, self).__init__()

        self.image_size = image_size

        self.conv1 = nn.Conv2d(in_channels=input_shape, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(kernel_size=2) # Reduces image to 25x25

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(kernel_size=2) # Reduces image to 12x12

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(kernel_size=2) # Reduces image to 6x6

        size = image_size
        for _ in range(3):
            size = (size - 2) // 2 + 1
        flatten_dim = 64 * size * size

        self.fc1 = nn.Linear(flatten_dim, mlp_units) # Fully Connected Block
        self.fc2 = nn.Linear(mlp_units, output_shape)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))

        x = x.view(x.size(0), -1)    # Flatten layer

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return torch.sigmoid(x)


# Conv model with Spatial Softmax
class SpatialSoftmax(nn.Module):
    def __init__(self, temperature: float = 1.0) -> None:
        super().__init__()
        self.temperature = temperature

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        b, h, w = logits.shape
        logits_flatten = logits.view(b, -1)
        probs_flatten = torch.softmax(logits_flatten / self.temperature, dim=1)
        probs = probs_flatten.view(b, h, w)
        return probs

class CoordinateRegressorSpatialSoftmax(nn.Module):
    """CNN architecture for predicting (x, y) coordinates of a pixel with spatial softmax."""

    def __init__(self, input_shape:int, output_shape:int, image_size: int = 50):
        super(CoordinateRegressorSpatialSoftmax, self).__init__()

        self.image_size = image_size

        self.conv1 = nn.Conv2d(in_channels=input_shape, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.logits_conv = nn.Conv2d(in_channels = 32, out_channels=1, kernel_size=1)
        self.spatial_softmax = SpatialSoftmax()

        x_coords = torch.arange(image_size, dtype=torch.float32)
        y_coords = torch.arange(image_size, dtype=torch.float32)

        grid_x, grid_y = torch.meshgrid(x_coords, y_coords, indexing='xy')
        self.register_buffer('grid_x', grid_x.clone())
        self.register_buffer('grid_y', grid_y.clone())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        logits = self.logits_conv(x)
        logits = logits.squeeze(1)
        probs = self.spatial_softmax(logits)
        x_pred = torch.einsum("bhw,hw->b", probs, self.grid_x)
        y_pred = torch.einsum("bhw,hw->b", probs, self.grid_y)
        output = torch.stack([x_pred, y_pred], dim=1) / 50.0
        return output


# performed poorly - not able to grasp the spatial structure.
class NeuralNetRegressor(nn.Module):
    def __init__(self, input_shape: int, mlp_units: int, output_shape: int, dropout_rate: float = 0.3) -> None:
        super(NeuralNetRegressor, self).__init__()
        self.fc1 = nn.Linear(input_shape * 50 * 50, mlp_units * 2)
        self.fc2 = nn.Linear(mlp_units * 2, mlp_units)
        self.fc3 = nn.Linear(mlp_units, output_shape)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        return torch.sigmoid(x)

## Train

In [37]:
DATASET_DIR = "Coordinate_Dataset"

def log_metrics(epoch, loss, accuracy, val_loss, val_accuracy):
    wandb.log({
        "epoch": epoch,
        "loss": loss,
        "accuracy": accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })


def save_checkpoint(epoch, run_name, model, loss, description):
    os.makedirs('checkpoints', exist_ok=True)
    checkpoint_path = os.path.join('checkpoints', f'{run_name}_{epoch + 1}.pt')
    torch.save({
        'epoch': epoch,
        'description': description,
        'model': model.state_dict(),
        'loss': loss,
    }, checkpoint_path)

def get_accuracy(
        ground_truth: np.ndarray,
        predictions: np.ndarray
) -> int:
    """
    This function gets the exact pixel accuracy for the outputs
    """
    ground_truth = (ground_truth * 50).round().int()
    predictions = (predictions * 50).round().int()
    accuracy = (ground_truth == predictions).all(dim=1)
    batch_accuracy = accuracy.float().mean()
    return batch_accuracy

def get_lenient_accuracy(
        ground_truth: torch.Tensor,
        predictions: torch.Tensor,
        threshold: float = 2.0
) -> float:
    """
    This adds a leniency of threshold pixels for the accuracy
    """
    gt_scaled = ground_truth * 50
    pred_scaled = predictions * 50

    distances = torch.sqrt(torch.sum((pred_scaled - gt_scaled) ** 2, dim=1))

    correct = (distances <= threshold).float().mean()
    return correct

def train(
        name: str,
        epochs: int,
        batch_size: int,
        learning_rate: float,
        model_name: str,
        early_stop: bool
):
    set_seed()
    wandb.init(project="CoordinatePrediction", name=name, config={
        "name": name,
        "epochs": epochs,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "description": "deepening the base model",
    })
    configs = wandb.config
    run_name = name

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    data_loaders = get_dataloaders(
        dataset_dir = DATASET_DIR,
        batch_size = configs.batch_size,
        shuffle = True,
        device = device
    )
    train_dataloader = data_loaders['train']
    val_dataloader = data_loaders['val']
    criterion = nn.MSELoss()

    if model_name == "simple_conv":
        coordinateregressor = CoordinateRegressor(input_shape=1, mlp_units=128, output_shape=2).to(device)
    elif model_name == "spatial_softmax":
        coordinateregressor = CoordinateRegressorSpatialSoftmax(input_shape=1, output_shape=2).to(device)
    else:
        coordinateregressor = NeuralNetRegressor(input_shape=1, mlp_units=128, output_shape=2).to(device)

    model_optim = torch.optim.AdamW(
        params = coordinateregressor.parameters(),
        lr = configs.learning_rate
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(model_optim, T_max=configs.epochs)

    prev_min_train_loss = 1e6            # train_loss for the previous minimum loss
    early_stoping_count = 10             # early stoping count for epochs with less than min loss
    count = 0
    earlystoping_threshold = 1e-6        # Threshold for early stoping counts to increase

    for epoch in tqdm(range(configs.epochs)):

        ##################### Training Loop #####################

        mean_loss, mean_accuracy = 0.0, 0.0

        for samples in train_dataloader:
            samples['image'], samples['coordinates'] = samples['image'].to(device), samples['coordinates'].to(device)
            coordinateregressor.train()
            output = coordinateregressor(samples['image'])
            ground_truth = samples['coordinates'].float()

            accuracy = get_accuracy(ground_truth, output)
            mean_accuracy += accuracy.item() / len(train_dataloader)

            model_optim.zero_grad()
            loss = criterion(output, ground_truth)
            loss.backward()
            model_optim.step()
            mean_loss += loss.item() / len(train_dataloader)
        scheduler.step()

        ##################### Validation Loop #####################
        mean_val_loss, mean_val_accuracy = 0.0, 0.0
        coordinateregressor.eval()
        with torch.inference_mode():
            for samples in val_dataloader:
                samples['image'], samples['coordinates'] = samples['image'].to(device), samples['coordinates'].to(device)
                output = coordinateregressor(samples['image'])
                ground_truth = samples['coordinates'].float()

                accuracy = get_accuracy(ground_truth, output)
                mean_val_accuracy += accuracy.item() / len(val_dataloader)

                loss = criterion(output, ground_truth)
                mean_val_loss += loss.item() / len(val_dataloader)

        ##################### Handling Early Stoping #####################
        if early_stop:
            if (prev_min_train_loss - mean_val_loss) <= earlystoping_threshold:
                count += 1
                if count >= early_stoping_count:
                    print(f"Stoping early epoch {epoch + 1} due to no major change in loss | train loss {mean_loss:.5f} train accuracy {mean_accuracy:.5f} val loss {mean_val_loss:.5f} val accuracy {mean_val_accuracy:.5f}")
                    save_checkpoint(epoch, run_name, coordinateregressor, loss, configs.description)
                    break
            else:
                count = 0
                prev_min_train_loss = min(prev_min_train_loss, mean_val_loss)

        ##################### logging & saving checkpoints #####################
        if epoch % 10 == 0:
            log_metrics(epoch, mean_loss, mean_accuracy, mean_val_loss, mean_val_accuracy)
            print(f"Epoch {epoch + 1} | mse loss: {mean_loss:.5f} | accuracy: {mean_accuracy:.5f}")
            print("#" * 50)
            print(f"validation = Epoch {epoch + 1} | mse loss: {mean_val_loss:.5f} | accuracy: {mean_val_accuracy:.5f}")
            save_checkpoint(epoch, run_name, coordinateregressor, loss, configs.description)

In [38]:
train(
    name = "spatial_softmax",
    epochs = 100,
    batch_size = 32,
    learning_rate = 0.001,
    model_name = "spatial_softmax",
    early_stop = True
   )

accuracy,▁██
epoch,▁▅█
loss,█▁▁
val_accuracy,▁██
val_loss,█▁▁
accuracy,0.91319
epoch,20
loss,2e-05
val_accuracy,0.90625
val_loss,3e-05


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1 | mse loss: 0.00142 | accuracy: 0.95139
##################################################
validation = Epoch 1 | mse loss: 0.00614 | accuracy: 0.03606
Epoch 11 | mse loss: 0.00000 | accuracy: 1.00000
##################################################
validation = Epoch 11 | mse loss: 0.00000 | accuracy: 1.00000
Stoping early epoch 12 due to no major change in loss | train loss 0.00000 train accuracy 1.00000 val loss 0.00000 val accuracy 1.00000


## Inference

In [39]:
class CoordinateInference:
    def __init__(self, model_path: str, device:str = 'cpu') -> None:
        self.device = device

        model_type = "_".join(model_path.split("/")[-1].split(".")[0].split("_")[:-1])
        if model_type == "simple_conv":
            self.model = CoordinateRegressor(input_shape=1, mlp_units=128, output_shape=2)
        elif model_type == "spatial_softmax":
            self.model = CoordinateRegressorSpatialSoftmax(input_shape=1, output_shape=2)
        else:
            self.model = NeuralNetRegressor(input_shape=1, mlp_units=128, output_shape=2)

        checkpoint = torch.load(model_path, map_location=self.device)
        state_dict = checkpoint['model']
        self.model.load_state_dict(state_dict)
        self.model.to(self.device)
        self.model.eval()

    def __call__(self, image_path: str) -> np.ndarray:
        image = Image.open(image_path)
        np_image = np.array(image, dtype=np.float32) / 255.0
        torch_image = torch.from_numpy(np_image).unsqueeze(0).unsqueeze(0)
        with torch.no_grad():
            output = self.model(torch_image).cpu().numpy()
        output = (output * 50).squeeze().round().astype(int)
        return output

def test_models(
        checkpoint_path: str,
        test_data_path: str,
        target_coordinate_path: str
) -> None:
    inference = CoordinateInference(checkpoint_path)
    with open(target_coordinate_path, "r") as f:
        target_coordinates = json.load(f)

    img_id_list = [images[:-4] for images in os.listdir(test_data_path)]
    accuracy = 0.0
    for id in tqdm(img_id_list):
        pred = inference(os.path.join(test_data_path, f"{id}.png"))
        gt = target_coordinates[id]
        accuracy += (pred == gt).all()
    accuracy /= len(os.listdir(test_data_path))
    print(f"test accuracy is {accuracy}")

In [41]:
id = '002267'
inference = CoordinateInference("checkpoints/spatial_softmax_12.pt")          # put the latest checpoint path
x_pred, y_pred = inference(f"Coordinate_Dataset/images/test/{id}.png")         # Put any image path to test model

with open("Coordinate_Dataset/target_coordinates.json", "r") as f:
    target_coordinates = json.load(f)

x, y = target_coordinates[id]
print(f"x coordinate = {x} | y coordinate = {y}\
        x pred coordinate = {x_pred} | y pred coordinate = {y_pred}")

x coordinate = 20 | y coordinate = 23        x pred coordinate = 20 | y pred coordinate = 23


## Testing

In [42]:
test_models(
    checkpoint_path = "checkpoints/spatial_softmax_12.pt",
    test_data_path = "Coordinate_Dataset/images/test",
    target_coordinate_path = "Coordinate_Dataset/target_coordinates.json"
)

  0%|          | 0/250 [00:00<?, ?it/s]

test accuracy is 0.996
